# Module 2 — Risk Monitoring (maize, GHA)
Stage-resolved **WRSI / WSI / crop-failure** water-balance monitoring (FAO-56/33) plus **SPI-3** meteorological drought — anchored on the planting dekad, masked to maize.

**Where to start:** put the `planting_pipeline` folder on your Google Drive, run the cells top-to-bottom, and approve the Drive-mount and Earth-Engine sign-in prompts.

## Setup

### Stage 0 · Runtime

**What runs.** Installs the Earth Engine Python client and `geemap` into the Colab runtime. Nothing is
computed here.

**Expected output.** One line, `installed.`, after 30 to 60 s on a cold runtime. Pip warnings about
dependency resolution are normal and can be ignored.

**If it fails.** Re-run the cell. A repeated failure usually means the runtime lost its network
connection; use *Runtime → Restart session* and start again.

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine sign-in

**What runs.** Connects to Earth Engine under the cloud project `PROJECT`. On a fresh runtime a
browser prompt appears; approve it with the Google account that has Earth Engine access.

**Expected output.** `EE ready: ok` within a few seconds. Anything else means the sign-in did not
complete.

**Which project to use.** Compute is identical across projects, but the **export queue is per
project**. `ee-manzikye` has stalled with tasks sitting in READY for hours. If you are going to
export, set `PROJECT = "indigo-proxy-484220-q8"` before running.

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Pipeline code on Drive

**What runs.** Mounts Google Drive and puts `/content/drive/MyDrive/planting_pipeline` on the Python
path, so `from src import ...` resolves to the pipeline modules rather than to anything installed by
pip.

**Expected output.** `Mounted at /content/drive` followed by
`pipeline on path: /content/drive/MyDrive/planting_pipeline`.

**If you get an `AssertionError`.** The folder is not where the cell expects it. Either upload the
whole `planting_pipeline` folder to the top level of My Drive, or edit `PIPE_DIR` to the real path.
The folder must contain `run.py`, `src/` and `config/`.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Stage 0d · Run configuration

**What you choose here.**

| Variable | Meaning |
|---|---|
| `COUNTRY`, `SEASON` | select a row of `config/season_calendar.csv`; this fixes the season window and the crop calendar |
| `YEAR` | the season's planting year. A season that crosses new year (short rains, Deyr) is still keyed by its planting year |
| `S1_ORBIT` | Sentinel-1 orbit. `ASCENDING` over Kenya, because Sentinel-1B failed in 2022 and descending coverage is sparse |
| `aoi` | the whole country, from the GAUL level-0 boundary |
| `aoi_run` | the area actually computed. It ships as a **test box**, 34.4 to 37.8 E and 1.2 S to 1.2 N, about 380 by 265 km over western and central Kenya |

**Time is counted in dekads, not dates.** A dekad is a third of a month, numbered 1 to 36 through the
year: dekad 1 is 1 to 10 January, dekad 9 is 21 to 31 March, dekad 36 is 21 to 31 December. Days 21 to
the month end are one dekad, so a dekad is 8, 9, 10 or 11 days long. `utils.dekad_label(9)` prints
`9·Mar`. Where a season crosses the new year the code uses a **global dekad** `gd` running 1 to 72,
which is the dekad of `YEAR` for 1 to 36 and of `YEAR + 1` for 37 to 72.

**Season windows this notebook can use.**

| Country · season | SOS detection window | Dekads |
|---|---|---|
| Kenya · Long rains | Mar-d3 to May-d3 | 9 to 15 |
| Kenya · Short rains | Oct-d1 to Nov-d3 | 28 to 33 |
| Ethiopia · Meher | Apr-d2 to Jun-d3 | 11 to 18 |

**Expected output.** One line, for example `Kenya · Long rains · 2024 · S1 ASCENDING`.

**Before you switch to the whole country.** Replace `aoi_run` with `aoi` only when the test box has
run cleanly. The country is roughly ten times the area, and Sentinel-2 and Sentinel-1 compositing
scales with it. Expect minutes to become tens of minutes, and expect `getInfo()` calls to time out;
at country scale use `ee.batch.Export` instead of reading results back into the notebook.

In [ ]:
# --- config + GEE-native map (geemap: built-in EE Layers panel, toggle + opacity) ---
COUNTRY="Kenya"      # "Kenya" | "Ethiopia"
SEASON ="Long rains" # "Long rains" | "Short rains" | "Meher"
YEAR=2024
S1_ORBIT="ASCENDING"   # S1B gone (2022) -> ASCENDING has coverage over Kenya
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=0).geometry()
aoi_run = ee.Geometry.Rectangle([34.4,-1.2,37.8,1.2])   # fast test box; use `aoi` for whole country
import geemap
try:
    from google.colab import output; output.enable_custom_widget_manager()  # needed for interactive geemap in Colab
except Exception:
    pass
def new_map(zoom=7):
    m = geemap.Map(add_google_map=False, basemap="SATELLITE")  # keyless Google tiles; native EE layer control
    m.centerObject(aoi_run, zoom)
    return m
def ee_layer(m, image, vis, name, shown=True, opacity=1.0):
    m.addLayer(ee.Image(image), vis, name, shown, opacity)  # appears in the Layers panel (toggle + opacity)
    return m
print(f"{COUNTRY} · {SEASON} · {YEAR} · S1 {S1_ORBIT}")

## Planting (onset anchor)

### Stage 1 · Planting dekad

**What this stage does.** It estimates, for every maize pixel, the dekad the crop was planted. Every
later module is anchored on this number, so an error here propagates into the water balance, the CPI
and the yield. Two different methods run, chosen by season.

**Main seasons: cue-fusion green-up.** Optical greenness is combined with radar so that cloud does not
leave holes. For each dekad a fused greenness proxy is built,

$$G_t=\tfrac{1}{2}\Big[\mathrm{unit}(\mathrm{NDRE}_t;0,0.7)+\mathrm{unit}(\mathrm{FPAR}_t;0,0.9)\Big],
\qquad G_t \leftarrow \mathrm{unit}(\mathrm{RVI}_t;0.1,0.8)\ \text{where optical is missing,}$$

where $\mathrm{unit}(x;a,b)$ rescales $x$ from $[a,b]$ to $[0,1]$. NDRE is the Sentinel-2 red-edge
index, FPAR is MODIS MCD15A3H, and RVI is the Sentinel-1 radar vegetation index, which rises with
canopy and is unaffected by cloud.

Start of season is the first dekad in the window at which greenness crosses a quarter of the season's
own amplitude and is still rising:

$$G_{\text{thr}}=G_{\min}+0.25\,(G_{\max}-G_{\min}),\qquad
\mathrm{SOS}=\min\{\,t:\ G_t\ge G_{\text{thr}}\ \wedge\ G_{t+1}-G_t\ge 0\ \wedge\ |t-\mathrm{SOS}_{\mathrm{LTN}}|\le 2\,\}$$

The last condition keeps the answer within two dekads of the climatological onset, which rejects weed
flushes and a second green-up. It is applied only where a climatology exists, so a sparse second-season
normal cannot reject every pixel.

Planting precedes visible green-up, so the detected SOS is shifted back by the crop's emergence lag:

$$\text{planting dekad} = \mathrm{SOS} - 2 \quad \text{(maize; wheat and teff use 1).}$$

**Short rains: rainfall onset.** Green-up detection is unreliable in the short rains, so the FEWS NET
rule is used instead. Onset is the first dekad with

$$P_t \ge 25\ \mathrm{mm}\quad\text{and}\quad P_{t+1}+P_{t+2}\ge 20\ \mathrm{mm}
\quad\text{and}\quad P_t/ET_{0,t}\ge 0.5 .$$

The first two conditions are the classic 25/20 mm rule; the third is an agroclimatic gate that asks
whether the rain was large relative to evaporative demand.

**Expected output.** A single line, `planting dekad computed for <country> <season>`. Nothing is
evaluated yet: Earth Engine is lazy, so errors in this cell often only surface at the next one, where
a number is actually requested.

**Expected values.** The result must fall inside the SOS window of the table above, minus the
emergence offset. For Kenya long rains 2024 the modal planting dekad is **8** (11 to 20 March), with
the 10th to 90th percentile of the 253 constituencies spanning dekads **7 to 9**. A modal dekad
outside 6 to 11 for that season means the fusion locked onto the wrong green-up.

In [ ]:
# --- planting dekad (onset) — cue-fusion green-up (main seasons) or rainfall onset (short rains) ---
from src import (utils, s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ,
                 ltn as LTN, planting_date as PD, wrsi_feedback as WR)
from run import crop_mask_image
kc, soil = utils.load_crop_coeffs()
rows={(r['country'],r['season']):r for r in utils.viable_products(utils.load_calendar('config/season_calendar.csv')) if r['crop'].lower()=='maize'}
r=rows[(COUNTRY,SEASON)]; ss,se=utils.sos_window_dekads(r['sos_detection_window']); mask=crop_mask_image(ee,COUNTRY,'maize',None)
if SEASON=='Short rains':
    pet=WR.pet_dekadal(ee,aoi_run,YEAR); ch=WR.chirps_dekadal(ee,aoi_run,YEAR)
    planting=WR.wrsi_onset(ee,ch,ss,se,pet_ic=pet).updateMask(mask).toInt16()
else:
    s2=S2.build_s2_dekadal(ee,aoi_run,YEAR); s1=S1.build_s1_dekadal(ee,aoi_run,YEAR,orbit=S1_ORBIT); fpar=FZ.add_fpar_dekadal(ee,aoi_run,YEAR)
    g=FZ.build_fused_greenness(ee,s2,s1,fpar); ltn=LTN.build_ltn_prior(ee,aoi_run,ss,se)
    sos=FZ.detect_sos(ee,g,mask,ss,se,ltn_sos=ltn,ltn_pad=2); planting=PD.sos_to_planting(ee,sos,'maize').toInt16()
print('planting dekad computed for', COUNTRY, SEASON)

### Stage 2 · Staged water balance: WRSI, WSI and crop failure

**What this stage does.** It runs a full FAO-56 and FAO-33 dekadal soil-water balance for every pixel,
starting at that pixel's own planting dekad, and reports how much of the crop's water requirement was
actually met. There is no hand-off to GeoWRSI; the balance is computed in Earth Engine.

**Reference evaporation, Hargreaves.** ERA5-Land daily 2 m temperature gives, per dekad,

$$ET_0 = 0.0023\,R_a\,(T_{\text{mean}}+17.8)\,\sqrt{T_{\max}-T_{\min}}\quad[\mathrm{mm\,d^{-1}}],$$

with extraterrestrial radiation $R_a$ computed per pixel from latitude and the mid-dekad day of year.
Hargreaves is used because it needs only temperature. ERA5-Land is used because GRIDMET does not cover
Africa.

**Crop water requirement.** The FAO-56 crop coefficient curve is stepped by dekads since planting: it
holds at $K_{c,\text{ini}}$ through the initial stage, ramps linearly to $K_{c,\text{mid}}$ through
development, holds, then ramps to $K_{c,\text{end}}$. For maize the pipeline uses
$K_c = 0.30 \rightarrow 1.20 \rightarrow 0.35$ over stages of 3, 4, 3 and 2 dekads, a 12-dekad cycle.
Then

$$WR_t = K_{c,t}\, ET_{0,t}.$$

**The balance.** Soil water starts empty at planting, which is the WRSI convention, and each dekad

$$W_t = SW_{t-1}+P_t,\qquad AET_t=\min(W_t,\,WR_t),\qquad
SW_t=\min\big(W_t-AET_t,\ WHC\big),$$

with rainfall $P$ from CHIRPS and the water-holding capacity $WHC$ from SoilGrids through the
Saxton and Rawls pedotransfer functions, integrated over a 1 m maize root zone. Water above $WHC$ is
discarded, which is exactly why this index cannot see waterlogging; that is what module 4 is for.

**The index.** Cumulated over the cycle,

$$\mathrm{WRSI}=100\,\frac{\sum_t AET_t}{\sum_t WR_t}.$$

The staged version snapshots the running WRSI at the end of each of the three stages, giving
`wrsi_veg`, `wrsi_flo` and `wrsi_grf`. The last is the whole-cycle value. `wsi_*` are the worst single
dekad of stress inside each stage.

**Classes (FEWS and GeoWRSI).**

| WRSI | Class |
|---|---|
| 95 to 100 | no or very mild deficit |
| 80 to 95 | mild |
| 60 to 80 | mediocre |
| 50 to 60 | poor |
| below 50 | **crop failure** |

`failflo` in this cell is the crop-failure flag at flowering, `wrsi_flo < 50`. Flowering is used because
FAO-33 makes it the stage where a deficit costs the most yield.

**Expected values.** In a normal Kenyan long rains, WRSI at flowering over maize sits between **70 and
100**, and the failure flag covers a small share of the area. An AOI-wide mean below 40, or a failure
flag over most of the map, almost always means the planting anchor is too early, so the balance is
being run through the dry weeks before the season.

**SPI-3.** The meteorological companion, a three-month standardised precipitation index from CHIRPS
against the 1981 to 2020 climatology. Earth Engine has no incomplete gamma function, so the
Wilson and Hilferty cube-root normal approximation is used:

$$a=\left(\frac{\mu}{\sigma}\right)^{2},\qquad
\mathrm{SPI}=\left[\left(\frac{P_3}{\mu}\right)^{1/3}-1+\frac{1}{9a}\right]\sqrt{9a}.$$

Classes follow McKee: **−1 moderate drought, −1.5 severe, −2 extreme**, and the wet mirror. By
construction SPI is roughly standard normal, so about 16 % of pixels below −1 in any given year is
normal. A map where most pixels are below −1 is a drought; a map where **all** of them are, including
the highlands, points at a CHIRPS gap rather than at weather.

In [ ]:
# --- staged WRSI/WSI + SPI-3 ---
from src.wrsi_waterbalance import run_wrsi_staged
from src import cpi as CPI, soil as SOIL, spi as SPI
mz=kc['maize']; d_veg=mz['L_ini']+mz['L_dev']; d_flo=d_veg+mz['L_mid']; lgp=mz['LGP_dekads']
whc=SOIL.get_whc(ee,aoi_run,soil,root_depth_cm=int(mz.get('root_depth_m',1.0)*100))
staged=run_wrsi_staged(ee,aoi_run,YEAR,planting,'maize',kc,soil,ss,se,whc_img=whc)
failflo=staged['wrsi_flo'].lt(50)                       # crop-failure at flowering (<50)
spi3=SPI.spi3(ee,aoi_run,YEAR,end_month=5 if SEASON=='Long rains' else (9 if SEASON=='Meher' else 12))
print('risk layers computed')

### Stage 3 · Fused canopy condition index

**What this stage does.** Provides a vegetation cross-check on the water balance. FCCI is the peak fused
greenness reached over the season,

$$\mathrm{FCCI}=100\times\max_{t\in\text{season}}G_t,$$

with the same fused $G$ as the planting stage, so it is cloud-proof through its radar fill and works at
10 to 20 m.

**Why it exists.** WRSI is a model: it says what the weather and the soil should have done to the crop.
FCCI is an observation: it says what the canopy actually looked like. When they disagree, something in
between is wrong, most often the planting date or irrigation the model does not know about.

**Expected values.** 0 to 100, higher being a more vigorous canopy. Vigorous rainfed maize peaks high,
failed or unplanted land stays low. Because $G$ uses fixed rescaling rather than a multi-year baseline,
values are comparable between pixels without an archive, but they are **not** an anomaly: a
consistently dry district looks low every year.

**Caveat to carry into any interpretation.** Where the peak dekad was cloudy, the value came from the
radar proxy, which is the less exact of the two cues.

In [ ]:
# --- Fused Canopy Condition Index (FCCI) — peak fused greenness (NDRE+FPAR+SAR), a vegetation cross-check on WRSI ---
from src import s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ
_s2=S2.build_s2_dekadal(ee,aoi_run,YEAR); _s1=S1.build_s1_dekadal(ee,aoi_run,YEAR,orbit=S1_ORBIT); _fp=FZ.add_fpar_dekadal(ee,aoi_run,YEAR)
fcci=FZ.fused_condition(ee, FZ.build_fused_greenness(ee,_s2,_s1,_fp), mask, ss, se, lgp=lgp)
print('FCCI computed (0-100; higher = more vigorous canopy)')

### Stage 4 · Map

**Layers, and how to read them together.**

| Layer | Range | Reading |
|---|---|---|
| WRSI at flowering | 40 to 100 | red is a deficit, green is satisfied demand |
| Crop failure at flowering | 0 or 1 | red where WRSI is below 50 |
| SPI-3 | −2 to 2 | red dry, blue wet, against 1981 to 2020 |
| Canopy condition, FCCI | 0 to 100 | red is a poor canopy, green is vigorous |

**The useful comparison is between layers, not within one.** Low WRSI with low FCCI is a real water
deficit that the crop felt. Low WRSI with high FCCI is usually irrigation, a spring, or a wrong planting
date. High WRSI with low FCCI points at a hazard the water balance cannot see, which is pests, disease,
flooding, or a field that was never planted.

In [ ]:
M=new_map()
ee_layer(M, staged['wrsi_flo'].updateMask(mask).clip(aoi_run), {'min':40,'max':100,'palette':['a50026','fee08b','1a9850']}, 'WRSI @flowering')
ee_layer(M, failflo.updateMask(mask).clip(aoi_run), {'min':0,'max':1,'palette':['ffffff','a50026']}, 'Crop-failure @ flowering')
ee_layer(M, spi3.updateMask(mask).clip(aoi_run), {'min':-2,'max':2,'palette':['a50026','fee08b','ffffff','abd9e9','4575b4']}, 'SPI-3 (drought/wet)')
ee_layer(M, fcci.clip(aoi_run), {'min':0,'max':100,'palette':['a50026','fee08b','1a9850']}, 'Canopy condition (fused, FCCI)')
M   # geemap renders its own GEE-native Layers panel (toggle + opacity slider) — no extra layer control needed

*WRSI < 50 = crop failure · SPI-3 ≤ −1 = drought · FCCI = 10–20 m cloud-proof canopy-vigour cross-check. Admin roll-up: `cpi_admin.py`, `spi_admin.py`, `fcci_admin.py`.*